In [1]:
import os
import re
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def chunk_text(text, chunk_size=512):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks



In [2]:
def ingest_pdfs(folder_path, chunk_size=512):

    corpus = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            raw_text = extract_text_from_pdf(pdf_path)
            cleaned_text = clean_text(raw_text)
            chunks = chunk_text(cleaned_text, chunk_size)

            for idx, chunk in enumerate(chunks, start=1):
                corpus.append({
                    "doc_id": filename,
                    "chunk_id": idx,
                    "text": chunk,
                    "metadata": {"source": filename}
                })
    return corpus


In [ ]:
folder = "./dataset"
data = ingest_pdfs(folder, chunk_size=512)
data

In [4]:
import json
import numpy as np
from tensorflow import keras

m = keras.models.load_model("modelo_skipgram.keras")

with open("enc.json", "r", encoding="utf-8") as f:
    enc = json.load(f)

with open("vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

enc

{'ingeniería': 0,
 'de': 1,
 'software,': 2,
 'en': 3,
 'su': 4,
 'novena': 5,
 'edición,': 6,
 'se': 7,
 'dirige': 8,
 'principalmente': 9,
 'a': 10,
 'estudiantes': 11,
 'universitarios': 12,
 'que': 13,
 'están': 14,
 'inscritos': 15,
 'cursos': 16,
 'tanto': 17,
 'introductorios': 18,
 'como': 19,
 'avanzados': 20,
 'software': 21,
 'y': 22,
 'sistemas.': 23,
 'asimismo,': 24,
 'los': 25,
 'ingenieros': 26,
 'trabajan': 27,
 'la': 28,
 'industria': 29,
 'encon-': 30,
 'trarán': 31,
 'el': 32,
 'libro': 33,
 'útil': 34,
 'lectura': 35,
 'general': 36,
 'para': 37,
 'actualizar': 38,
 'sus': 39,
 'conocimientos': 40,
 'acerca': 41,
 'temas': 42,
 'reutilización': 43,
 'diseño': 44,
 'arquitectónico,': 45,
 'confiabilidad,': 46,
 'seguridad': 47,
 'mejora': 48,
 'procesos.': 49,
 'presente': 50,
 'obra': 51,
 'actualizó': 52,
 'al': 53,
 'incorporar': 54,
 'siguientes': 55,
 'cambios:': 56,
 '>': 57,
 'nuevos': 58,
 'capítulos': 59,
 'sobre': 60,
 'ágil': 61,
 'sistemas': 62,
 'embebi

In [5]:
w = m.get_weights()[0]

words_emb = {word: w[enc[word]] for word in vocabulary}

words_emb

{'ingeniería': array([-0.31392562, -0.8084513 , -0.60121006, -0.19005565, -0.7490297 ,
         0.33619726, -0.40175587, -0.19827767, -0.29375148, -0.45581937,
         0.5786906 , -0.39940223,  0.00351342,  0.3951092 , -0.5925517 ,
         0.09032252, -0.17742595, -0.13616203,  0.31217483,  0.62910175,
         0.32371455, -0.3468425 ,  0.28346843, -0.05679432,  0.01770537,
        -0.42016816, -0.0752369 , -0.32220083, -0.41257113, -0.53802   ,
         0.28850603, -0.26942134,  0.7953567 ,  0.3876676 , -0.854521  ,
        -0.07773022,  0.3892008 ,  0.16070046,  0.48359135, -0.18053268,
        -0.11119825,  0.29173326, -0.36704206,  0.5921323 , -0.16285266,
        -0.26251414, -0.22615978, -0.31350663, -0.40955895,  0.30748853,
         0.04354192, -0.33894044, -0.0404931 , -0.30471864,  0.3088621 ,
         0.5045373 , -0.56211764,  0.44773665, -0.04898641, -0.19286862,
         0.2829972 , -0.52960783,  0.19159217,  0.07277896, -0.00941564,
        -0.7371788 , -0.43621355, -0.

In [6]:
def chunk_embedding(chunk, words_emb, enc, dim=200):
    vectors = []
    for word in chunk.split():
        if word in enc:
            vectors.append(words_emb[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(dim) 

In [9]:
chunk = "el perro ladra"
vector = chunk_embedding(chunk, words_emb, enc)
vector

array([-0.01386252,  0.13889009, -0.14433935,  0.14909221,  0.11329032,
       -0.20618026,  0.2654145 ,  0.11330456, -0.20350833,  0.03564827,
       -0.0921599 , -0.04038884,  0.189936  , -0.11553169, -0.01387334,
        0.18322057, -0.1895919 , -0.0189737 ,  0.20965722,  0.23534533,
        0.08310977,  0.26108706,  0.09157807, -0.12320127, -0.12519109,
        0.03343531,  0.01400776, -0.11634532,  0.14701445,  0.2021763 ,
       -0.2526449 , -0.20827341, -0.17439173,  0.07903011,  0.27020568,
       -0.09737261, -0.02154347, -0.01709224,  0.17699614, -0.16276553,
       -0.11299948,  0.1669688 ,  0.1162015 , -0.30655816,  0.03921711,
       -0.23018071, -0.22567096, -0.18232901, -0.0448006 , -0.00359121,
        0.01793439,  0.09646232,  0.1060081 , -0.10088427,  0.05095683,
       -0.01335164, -0.12826888,  0.00930758,  0.27148244, -0.12322957,
        0.04466404,  0.21522754, -0.11368878, -0.02921087,  0.06351046,
        0.10061301,  0.05986971,  0.02454288,  0.07398757,  0.05

In [10]:
def generate_embeddings(corpus, words_emb, enc, dim=200): 
    results = [] 
    
    for item in corpus: 
        vector = chunk_embedding(item["text"], words_emb, enc, dim) 
        results.append({ 
            "doc_id": item["doc_id"], 
            "chunk_id": item["chunk_id"], 
            "embedding": vector.tolist(), 
            "metadata": item["metadata"] 
        })
    return results

In [12]:
embeddings = generate_embeddings(data, words_emb, enc)
embeddings

[{'doc_id': 'C1.pdf',
  'chunk_id': 1,
  'embedding': [-0.06979870051145554,
   -0.13498148322105408,
   -0.06504863500595093,
   0.08865693211555481,
   -0.06874031573534012,
   -0.004137716721743345,
   0.03426751494407654,
   0.002649934496730566,
   0.0846395194530487,
   0.047786444425582886,
   0.03906698524951935,
   0.05055045709013939,
   0.005900396965444088,
   -0.10052550584077835,
   0.004693911410868168,
   0.05801450461149216,
   0.04519264027476311,
   -0.009289263747632504,
   0.009963730350136757,
   0.025615058839321136,
   -0.017616914585232735,
   0.11161138117313385,
   -0.081707663834095,
   -0.02570279873907566,
   -0.0960691049695015,
   -0.03598008677363396,
   0.08559989929199219,
   0.03806223347783089,
   -0.07180196046829224,
   0.04673251882195473,
   -0.1298297643661499,
   -0.04868985339999199,
   0.08117377012968063,
   0.11987462639808655,
   -0.038152698427438736,
   0.02224585972726345,
   0.039571695029735565,
   -0.08628186583518982,
   -0.0377678

In [13]:
import psycopg2
import json
import numpy as np

conn = psycopg2.connect(
    dbname="Tesis",   
    user="postgres",  
    password="1234",
    host="localhost",
    port="5432"      
)
cursor = conn.cursor()

In [18]:
def insert_embedding(doc_id, chunk_id, embedding, metadata):
    try:
        cursor.execute("""
            INSERT INTO embeddings (doc_id, chunk_id, embedding, metadata)
            VALUES (%s, %s, %s, %s)
        """, (
            doc_id,
            chunk_id,
            json.dumps(embedding.tolist()),
            json.dumps(metadata)
        ))
        conn.commit()
    except Exception as e:
        print("Error al insertar:", e)
        conn.rollback() 


In [21]:
def populate_embeddings(embeddings):
    for item in embeddings:
        insert_embedding(
            item["doc_id"],
            item["chunk_id"],
            np.array(item["embedding"], dtype="float32"),
            item["metadata"]
        )

populate_embeddings(embeddings)
print("Embeddings insertados correctamente en la base de datos.")

Embeddings insertados correctamente en la base de datos.


In [34]:
def get_all_embeddings():
    cursor.execute("SELECT doc_id, chunk_id, embedding, metadata FROM embeddings")
    rows = cursor.fetchall()
    data = []
    for doc_id, chunk_id, emb, meta in rows:
        # Si la columna embedding es JSONB
        if isinstance(emb, str):
            emb = np.array(json.loads(emb), dtype="float32")
        # Si la columna embedding es float8[]
        elif isinstance(emb, list):
            emb = np.array(emb, dtype="float32")
        data.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "embedding": emb,
            "metadata": json.loads(meta) if isinstance(meta, str) else meta
        })
    return data

registros = get_all_embeddings()
registros


[{'doc_id': 'C1.pdf',
  'chunk_id': 1,
  'embedding': array([-0.0697987 , -0.13498148, -0.06504864,  0.08865693, -0.06874032,
         -0.00413772,  0.03426751,  0.00264993,  0.08463952,  0.04778644,
          0.03906699,  0.05055046,  0.0059004 , -0.10052551,  0.00469391,
          0.0580145 ,  0.04519264, -0.00928926,  0.00996373,  0.02561506,
         -0.01761691,  0.11161138, -0.08170766, -0.0257028 , -0.0960691 ,
         -0.03598009,  0.0855999 ,  0.03806223, -0.07180196,  0.04673252,
         -0.12982976, -0.04868985,  0.08117377,  0.11987463, -0.0381527 ,
          0.02224586,  0.0395717 , -0.08628187, -0.03776781,  0.05775293,
         -0.03650211,  0.03002232,  0.00393053, -0.04778845, -0.04677264,
         -0.06573744, -0.06955735,  0.03021413, -0.01188462,  0.00479936,
         -0.08991595,  0.01281324, -0.00284607, -0.03275149,  0.00622339,
         -0.01443374, -0.02243453, -0.03215268,  0.04084194, -0.0390839 ,
         -0.08033197,  0.05795707, -0.07744037, -0.00782393,

In [38]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query_text, k=5):
    query_vec = chunk_embedding(query_text, words_emb, enc)
    registros = get_all_embeddings()
    scores = []

    for r in registros:
        sim = cosine_similarity(query_vec, r["embedding"])
        scores.append((sim, r))
    scores.sort(reverse=True, key=lambda x: x[0])

    return scores[:k]

resultados = retrieve("¿Qué es la ingenieria de software?", k=3)
resultados


[(0.36262903,
  {'doc_id': 'C1.pdf',
   'chunk_id': 3,
   'embedding': array([-0.12575829, -0.10150015, -0.09758019,  0.07107665, -0.09818219,
          -0.01034999,  0.10052368, -0.01088765,  0.04023251,  0.00545238,
           0.03934193,  0.07441415,  0.03823575, -0.10002116, -0.03281178,
           0.04944971,  0.01511002, -0.02035744,  0.06672146,  0.06211837,
           0.00863377,  0.12726782, -0.05137251,  0.02266085, -0.07930761,
          -0.00471078,  0.06135564,  0.06001158, -0.02843001,  0.05185909,
          -0.14545086, -0.1114347 ,  0.08876333,  0.04023084,  0.00403046,
           0.0532358 ,  0.03674607, -0.13193052, -0.01051319,  0.02271941,
          -0.02764128,  0.09671049, -0.01522501, -0.04563518, -0.04711944,
          -0.05092902, -0.08697946, -0.03478217, -0.06650165, -0.02765392,
          -0.10875075,  0.02966444, -0.05076781, -0.04346792,  0.00296469,
           0.01119767, -0.03954016,  0.00430411,  0.01985091, -0.0613362 ,
          -0.10086472,  0.102177